# 00 Part 1 — Data Preparation (per BRAND_BUCKET)

**Input:**
- `data/{BRAND_BUCKET}/amazon_shoes_8_combined.parquet` (created by `brand_split/00_brand_filter.ipynb`)
- `code/{BRAND_BUCKET}/asin_split.csv` (created by `brand_split/01_stratified_split.ipynb`)

**Output:**
```
data/{BRAND_BUCKET}/amzn_shoes_monthly_diffs_ffill_fixed_splits/
    all_products_clean.parquet         ← all clean ASINs for this bucket
    train-00000-of-00001.parquet       ← subset by asin_split.csv (split=train)
    validation-00000-of-00001.parquet  ← subset by asin_split.csv (split=val)
data/{BRAND_BUCKET}/keys/
    main_train_keys.csv                ← (ASIN, date) pairs for train
    main_val_keys.csv                  ← (ASIN, date) pairs for val
    asin_split.csv                     ← copied from brand_split/01
```

Parameterized by `BRAND_BUCKET` — set it in cell ② and re-run for each of the 6 buckets:
- `women_top1_dream_pairs`, `women_top2_idifu`, `women_other`
- `men_top1_bruno_marc`, `men_top2_nike`, `men_other`

**Pipeline:**
1. Load → remove dirty ASINs → filter to expected gender
2. Drop dead columns → fix dtypes → rebuild subcat (top 10 auto-derived per bucket)
3. Forward fill on FULL data (Jan–Mar) — Jan/Feb acts as seed for March
4. Create NaN flags on trimmed window only
5. Trim to Mar 2025 onward → drop incomplete ASINs (show list first)
6. Final checks → save `all_products_clean.parquet` → removal log
7. **Apply pre-computed stratified split** → save train/val parquets + key CSVs

## ① Setup

In [40]:
!pip install pyarrow pandas -q
print('✅ Done')

✅ Done


## ② Config

In [41]:
import os, shutil
import pandas as pd
import numpy as np
import random
from pathlib import Path

def _find_project_root() -> Path:
    """Walk up from cwd until a directory containing both
    'data_preparation/' and 'code/' is found.
    Works whether Jupyter/PyCharm sets cwd to project root or notebook dir."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

# ── BRAND BUCKET — change this line for each run ──────────────────────────
# Valid values (created by brand_split/00_brand_filter.ipynb):
#   women_top1_dream_pairs, women_top2_idifu, women_other,
#   men_top1_bruno_marc,    men_top2_nike,    men_other
BRAND_BUCKET = 'men_adidas'

# Derive expected gender from the bucket name
if BRAND_BUCKET.startswith('women'):
    EXPECTED_GENDER = 'Women'
elif BRAND_BUCKET.startswith('men'):
    EXPECTED_GENDER = 'Men'
else:
    raise ValueError(f'BRAND_BUCKET must start with "women" or "men": {BRAND_BUCKET}')

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data' / BRAND_BUCKET
DATA_DIR     = DATA_ROOT / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
IMG_DIR      = PROJECT_ROOT / 'data' / 'images'      # shared across buckets (ASIN-unique)
PRED_DIR     = DATA_ROOT / 'predictions'
KEY_DIR      = DATA_ROOT / 'keys'

INPUT_FILE   = DATA_ROOT / 'amazon_shoes_8_combined.parquet'
CLEAN_FILE   = DATA_DIR  / 'all_products_clean.parquet'

assert INPUT_FILE.exists(), (
    f'Missing input file: {INPUT_FILE}\n'
    f'Did you run brand_split/00_brand_filter.ipynb first?'
)

os.makedirs(DATA_DIR, exist_ok=True)

# ── Window ──────────────────────────────────────────────────────────────────
START_DATE = '2025-03-01'
END_DATE   = '2026-03-31'

# ── Columns ─────────────────────────────────────────────────────────────────
FFILL_COLS = ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']

DROP_COLS = [
    'SALES_RANK_original', 'PRICE_original', 'BUYBOX_PRICE_original',
    'gender', 'manufacturer', 'brand', 'model', 'color', 'size',
]

REQUIRED_COMPLETE = ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']

NAN_TOLERANCE = 0.0

PAPER_SCHEMA = {
    'ASIN': 'object', 'window': 'float64', 'date': 'datetime64[ns]',
    'SALES_RANK': 'float64', 'PRICE': 'float64', 'BUYBOX_PRICE': 'float64',
    'text': 'object', 'RATING': 'float64', 'REVIEW_COUNT': 'float64',
    'subcat': 'object', 'subcat_aggregated': 'object',
    'New Offer Count: Current': 'int64',
    'Count of retrieved live offers: New, FBA': 'int64',
    'Count of retrieved live offers: New, FBM': 'int64',
    'Lightning Deals: Upcoming Deal': 'int64',
    'Buy Box: Is FBA': 'int64',
    'image': 'object',
}

removal_log = []
def log_removal(asins, reason):
    for asin in asins:
        removal_log.append({'ASIN': asin, 'reason': reason})

print('✅ Config ready')
print(f'  BRAND_BUCKET    : {BRAND_BUCKET}')
print(f'  EXPECTED_GENDER : {EXPECTED_GENDER}')
print(f'  Input           : {INPUT_FILE}')
print(f'  Output dir      : {DATA_DIR}')
print(f'  Clean file      : {CLEAN_FILE}')
print(f'  Window          : {START_DATE} → {END_DATE}')
print(f'  Fill seed       : Jan + Feb 2025 (trimmed after fill)')
print(f'  NaN tolerance   : {NAN_TOLERANCE*100:.0f}% (0% = any NaN drops ASIN)')
print()
print(f'  Split file      : {KEY_DIR / "asin_split.csv"}  (from brand_split/01)')

✅ Config ready
  BRAND_BUCKET    : men_adidas
  EXPECTED_GENDER : Men
  Input           : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_adidas/amazon_shoes_8_combined.parquet
  Output dir      : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits
  Clean file      : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits/all_products_clean.parquet
  Window          : 2025-03-01 → 2026-03-31
  Fill seed       : Jan + Feb 2025 (trimmed after fill)
  NaN tolerance   : 0% (0% = any NaN drops ASIN)

  Split file      : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_adidas/keys/asin_split.csv  (from brand_split/01)


## ③ Load

In [42]:
df = pd.read_parquet(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

# Save subcat map before any filtering — used later in removal log
asin_subcat_map = df.drop_duplicates('ASIN').set_index('ASIN')['subcat'].to_dict()

print(f'Shape        : {df.shape}')
print(f'Unique ASINs : {df["ASIN"].nunique():,}')
print(f'Date range   : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print('Gender (ASIN level):')
print(df.drop_duplicates('ASIN')['gender'].value_counts().to_string())

Shape        : (10270, 26)
Unique ASINs : 79
Date range   : 2025-01-06 → 2026-03-30

Gender (ASIN level):
gender
Men    79


## ④ Remove Dirty ASINs — Wrong Gender, Wrong Size, 100% NaN

In [43]:
# ── Wrong gender / wrong size ─────────────────────────────────────────────
BAD_GENDERS = ['Costumes & Accessories', 'Shoe, Jewelry & Watch Accessories']
gender_size_asins = df[
    df['gender'].isin(BAD_GENDERS) | (df['size'] != '8')
]['ASIN'].unique().tolist()

print(f'Wrong gender/size ASINs ({len(gender_size_asins)}):')
for asin in sorted(gender_size_asins):
    g = df[df['ASIN']==asin]['gender'].iloc[0]
    s = df[df['ASIN']==asin]['size'].iloc[0]
    print(f'  {asin}  gender={g}  size={s}')
log_removal(gender_size_asins, 'wrong gender or size')

# ── 100% NaN in any critical column (checked on EXPECTED_GENDER only) ────
CRITICAL_COLS = ['SALES_RANK', 'PRICE', 'RATING', 'REVIEW_COUNT']
df_gender_temp = df[
    (df['gender'] == EXPECTED_GENDER) & (~df['ASIN'].isin(gender_size_asins))
].copy()

print(f'\nChecking {df_gender_temp["ASIN"].nunique():,} {EXPECTED_GENDER} ASINs for 100% NaN...')
all_nan_asins = set()
for col in CRITICAL_COLS:
    fully_nan = df_gender_temp.groupby('ASIN')[col].apply(lambda x: x.isna().all())
    flagged   = fully_nan[fully_nan].index.tolist()
    if flagged:
        print(f'  {col} — {len(flagged)} ASIN(s) 100% NaN:')
        for asin in sorted(flagged):
            subcat = df_gender_temp[df_gender_temp['ASIN']==asin]['subcat'].iloc[0]
            print(f'    {asin}  subcat={subcat}')
        all_nan_asins.update(flagged)
    else:
        print(f'  {col} — no 100% NaN ASINs ✅')
log_removal(list(all_nan_asins), '100% NaN in critical column')

dirty_asins = list(set(gender_size_asins) | all_nan_asins)
print(f'\nTotal dirty ASINs to remove: {len(dirty_asins)}')

Wrong gender/size ASINs (0):

Checking 79 Men ASINs for 100% NaN...
  SALES_RANK — no 100% NaN ASINs ✅
  PRICE — no 100% NaN ASINs ✅
  RATING — no 100% NaN ASINs ✅
  REVIEW_COUNT — no 100% NaN ASINs ✅

Total dirty ASINs to remove: 0


## ⑤ Filter to EXPECTED_GENDER + Remove Dirty

In [44]:
before = df['ASIN'].nunique()
df = df[(df['gender'] == EXPECTED_GENDER) & (~df['ASIN'].isin(dirty_asins))].copy()
print(f'ASINs : {before:,} → {df["ASIN"].nunique():,}  (kept {EXPECTED_GENDER}, removed dirty)')
print(f'Rows  : {len(df):,}')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print('Window distribution:')
print(df['window'].value_counts().to_string())

ASINs : 79 → 79  (kept Men, removed dirty)
Rows  : 10,270
Date range : 2025-01-06 → 2026-03-30

Window distribution:
window
7     5135
28    5135


## ⑥ Drop Dead Columns — Early to Reduce Memory

In [45]:
cols_dropped = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=cols_dropped)
print(f'Dropped {len(cols_dropped)} columns: {cols_dropped}')
print(f'Remaining ({len(df.columns)}): {df.columns.tolist()}')

Dropped 9 columns: ['SALES_RANK_original', 'PRICE_original', 'BUYBOX_PRICE_original', 'gender', 'manufacturer', 'brand', 'model', 'color', 'size']
Remaining (17): ['ASIN', 'window', 'date', 'SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'text', 'RATING', 'REVIEW_COUNT', 'subcat', 'subcat_aggregated', 'New Offer Count: Current', 'Count of retrieved live offers: New, FBA', 'Count of retrieved live offers: New, FBM', 'Lightning Deals: Upcoming Deal', 'Buy Box: Is FBA', 'image']


## ⑦ Fix Dtypes

In [46]:
df['window'] = df['window'].astype('float64')
df['date']   = pd.to_datetime(df['date'])

# FBA and FBM — fill NaN with 0
# Keepa did not return offer data = treat as 0 active offers that week
for col in ['Count of retrieved live offers: New, FBA',
            'Count of retrieved live offers: New, FBM']:
    if col in df.columns:
        n = df[col].isna().sum()
        df[col] = df[col].fillna(0).astype('int64')
        short = col.split(': ')[-1]
        print(f'  {short:<8} filled {n:,} NaN → 0  (no Keepa offer data = 0 offers)')

df['New Offer Count: Current']       = df['New Offer Count: Current'].fillna(0).astype('int64')
df['Lightning Deals: Upcoming Deal'] = df['Lightning Deals: Upcoming Deal'].astype('int64')
df['Buy Box: Is FBA']                = df['Buy Box: Is FBA'].astype('int64')
print('✅ Dtypes fixed')

  New, FBA filled 9,268 NaN → 0  (no Keepa offer data = 0 offers)
  New, FBM filled 9,256 NaN → 0  (no Keepa offer data = 0 offers)
✅ Dtypes fixed


## ⑧ Rebuild subcat_aggregated — Top 10 Subcats in This Bucket + Other

In [47]:
# Auto-derive top 10 subcats from THIS bucket's data.
# A hardcoded list (e.g. Women full-set top10) would collapse most rows to
# "Other" for narrow buckets like men_top2_nike — re-derive per bucket.
top10_subcats = (
    df.drop_duplicates('ASIN')['subcat']
    .value_counts()
    .head(10)
    .index.tolist()
)
TOP_10 = set(top10_subcats)

print(f'TOP_10 (auto-derived for {BRAND_BUCKET}):')
for i, s in enumerate(top10_subcats, 1):
    n = (df.drop_duplicates('ASIN')['subcat'] == s).sum()
    print(f'  {i:>2}. {s:<35} {n:>4} ASINs')
print()

df['subcat_aggregated'] = df['subcat'].apply(lambda x: x if x in TOP_10 else 'Other')

result = (
    df.drop_duplicates('ASIN')
    .groupby('subcat_aggregated')['ASIN'].count()
    .sort_values(ascending=False).reset_index()
    .rename(columns={'ASIN': 'asin_count'})
)
result['pct'] = (result['asin_count'] / result['asin_count'].sum() * 100).round(1)
print(result.to_string(index=False))
print(f'\nTotal: {result["asin_count"].sum():,} ASINs | {result.shape[0]} groups')

TOP_10 (auto-derived for men_adidas):
   1. Golf                                  51 ASINs
   2. Road Running                          10 ASINs
   3. Wrestling                              6 ASINs
   4. Fashion Sneakers                       6 ASINs
   5. Fitness & Cross-Training               2 ASINs
   6. Football                               1 ASINs
   7. Skateboarding                          1 ASINs
   8. Tennis & Racquet Sports                1 ASINs
   9. Basketball                             1 ASINs

       subcat_aggregated  asin_count  pct
                    Golf          51 64.6
            Road Running          10 12.7
        Fashion Sneakers           6  7.6
               Wrestling           6  7.6
Fitness & Cross-Training           2  2.5
              Basketball           1  1.3
                Football           1  1.3
           Skateboarding           1  1.3
 Tennis & Racquet Sports           1  1.3

Total: 79 ASINs | 9 groups


## ⑨ Fill Missing Values — On Full Data Including Jan/Feb

**This is the critical step.**
Jan and Feb 2025 act as the seed for the March window.
We fill NOW, before trimming, so March inherits valid values from earlier months.
BUYBOX_PRICE also fills any leading gaps using the first known price in the window.

In [48]:
df = df.sort_values(['ASIN', 'window', 'date']).reset_index(drop=True)
fill_cols = [c for c in FFILL_COLS if c in df.columns]

print('NaN counts BEFORE fill (full Jan–Mar data):')
for col in fill_cols:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  ({n/len(df)*100:.2f}%)')

# Fill all columns forward per (ASIN, window)
# BUYBOX_PRICE additionally fills leading NaN using the first known value
# — products that gained a Buy Box during the window had no prior price data
other_cols  = [c for c in fill_cols if c != 'BUYBOX_PRICE']
buybox_cols = ['BUYBOX_PRICE'] if 'BUYBOX_PRICE' in fill_cols else []

if other_cols:
    df[other_cols] = (
        df.groupby(['ASIN', 'window'])[other_cols]
        .transform(lambda s: s.ffill())
    )

if buybox_cols:
    df['BUYBOX_PRICE'] = (
        df.groupby(['ASIN', 'window'])['BUYBOX_PRICE']
        .transform(lambda s: s.ffill().bfill())
    )

print('\nNaN counts AFTER fill:')
for col in fill_cols:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  ({n/len(df)*100:.2f}%)')

NaN counts BEFORE fill (full Jan–Mar data):
  SALES_RANK                          : 378  (3.68%)
  PRICE                               : 771  (7.51%)
  BUYBOX_PRICE                        : 1,517  (14.77%)
  RATING                              : 484  (4.71%)
  REVIEW_COUNT                        : 506  (4.93%)

NaN counts AFTER fill:
  SALES_RANK                          : 300  (2.92%)
  PRICE                               : 449  (4.37%)
  BUYBOX_PRICE                        : 130  (1.27%)
  RATING                              : 453  (4.41%)
  REVIEW_COUNT                        : 475  (4.63%)


## ⑩ Trim to Analysis Window, Then Create NaN Flags

Trim Jan/Feb NOW — after fill is done.
Create NaN flags on the trimmed window only, so they accurately reflect
which values were originally missing in the March→Mar 2026 period.

In [49]:
before_rows = len(df)
df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)].copy()

print(f'Rows  : {before_rows:,} → {len(df):,}  (removed {before_rows-len(df):,} Jan/Feb rows)')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'ASINs : {df["ASIN"].nunique():,}')
print()

# NaN check on first date — should be near zero now
first_date = df['date'].min()
fw = df[df['date'] == first_date][fill_cols]
print(f'First date NaN check ({first_date.date()}) — should be near zero after fill:')
for col in fill_cols:
    n = fw[col].isna().sum()
    pct = n / len(fw) * 100
    flag = '✅' if pct < 1 else ('🟡' if pct < 10 else '🔴')
    print(f'  {col:<35} : {n:,} / {len(fw):,}  ({pct:.1f}%)  {flag}')

print()
# Create NaN flags on trimmed window
for col in fill_cols:
    if col in df.columns:
        df[f'is_{col}_filled'] = df[col].isna().astype(int)

print('NaN flag columns created on trimmed window:')
for col in fill_cols:
    flag = f'is_{col}_filled'
    n   = df[flag].sum()
    pct = n / len(df) * 100
    print(f'  {flag:<30} : {n:,} rows were NaN ({pct:.2f}%)')

Rows  : 10,270 → 9,006  (removed 1,264 Jan/Feb rows)
Date range : 2025-03-03 → 2026-03-30
ASINs : 79

First date NaN check (2025-03-03) — should be near zero after fill:
  SALES_RANK                          : 9 / 158  (5.7%)  🟡
  PRICE                               : 15 / 158  (9.5%)  🟡
  BUYBOX_PRICE                        : 2 / 158  (1.3%)  🟡
  RATING                              : 17 / 158  (10.8%)  🔴
  REVIEW_COUNT                        : 17 / 158  (10.8%)  🔴

NaN flag columns created on trimmed window:
  is_SALES_RANK_filled           : 121 rows were NaN (1.34%)
  is_PRICE_filled                : 244 rows were NaN (2.71%)
  is_BUYBOX_PRICE_filled         : 114 rows were NaN (1.27%)
  is_RATING_filled               : 211 rows were NaN (2.34%)
  is_REVIEW_COUNT_filled         : 211 rows were NaN (2.34%)


## ⑪ Find Incomplete ASINs — Show List Before Dropping

An ASIN is incomplete if any of its rows in the analysis window still have NaN
after fill. These cannot be fixed — the ASIN had no data to carry forward even
from Jan/Feb.

`NAN_TOLERANCE = 0.0` means strictly zero NaN allowed (any NaN = drop).
Change this in Config if you want to allow a small % of missing rows.

In [50]:
all_incomplete = {}

print('Checking for ASINs with remaining NaN after fill...')
print()

for col in REQUIRED_COMPLETE:
    # BUYBOX_PRICE: only drop if 100% missing — partial gaps were already filled
    # All other columns: use NAN_TOLERANCE (default 0 = any NaN = drop)
    threshold = 0.999 if col == 'BUYBOX_PRICE' else NAN_TOLERANCE

    nan_per_asin = (
        df[df['window'] == 7.0]
        .groupby('ASIN')[col]
        .apply(lambda x: (x.isna().sum() / len(x)) > threshold)
    )
    affected = nan_per_asin[nan_per_asin].index.tolist()
    all_incomplete[col] = affected

    if not affected:
        print(f'{col:<35} : no incomplete ASINs ✅')
    else:
        print(f'{col:<35} : {len(affected)} ASINs flagged:')
        for asin in sorted(affected)[:20]:
            subcat    = df[df['ASIN']==asin]['subcat'].iloc[0]
            nan_rows  = df[(df['ASIN']==asin) & df[col].isna()].shape[0]
            total_rows = df[df['ASIN']==asin].shape[0]
            pct_nan   = nan_rows / total_rows * 100
            print(f'  {asin}  subcat={subcat:<35}  NaN={nan_rows}/{total_rows} ({pct_nan:.0f}%)')
        if len(affected) > 20:
            print(f'  ... and {len(affected)-20} more')
    print()

asins_to_drop = set()
for asins in all_incomplete.values():
    asins_to_drop.update(asins)

print(f'Total unique ASINs to drop: {len(asins_to_drop):,}')
pct_lost = len(asins_to_drop) / df['ASIN'].nunique() * 100
print(f'That is {pct_lost:.1f}% of the dataset')
print()
if pct_lost > 10:
    print('⚠️  More than 10% of ASINs would be dropped — review the list above.')
else:
    print('✅ Within acceptable range — safe to drop.')

Checking for ASINs with remaining NaN after fill...

SALES_RANK                          : 5 ASINs flagged:
  B0C62CZ47J  subcat=Golf                                 NaN=19/114 (17%)
  B0CKRZM9PN  subcat=Road Running                         NaN=41/114 (36%)
  B0D2S77TY9  subcat=Basketball                           NaN=1/114 (1%)
  B0DJTSDSBV  subcat=Fitness & Cross-Training             NaN=19/114 (17%)
  B0DJTYB32Z  subcat=Road Running                         NaN=41/114 (36%)

PRICE                               : 8 ASINs flagged:
  B07T391K1H  subcat=Football                             NaN=65/114 (57%)
  B08CZ3Z52Q  subcat=Road Running                         NaN=9/114 (8%)
  B0CYR2Z2P1  subcat=Fashion Sneakers                     NaN=73/114 (64%)
  B0CZ74HSD4  subcat=Fashion Sneakers                     NaN=18/114 (16%)
  B0D17WQJG2  subcat=Fashion Sneakers                     NaN=10/114 (9%)
  B0D2S77TY9  subcat=Basketball                           NaN=1/114 (1%)
  B0DJTSDSBV  subc

## ⑫ Execute Drop

In [51]:
for col, asins in all_incomplete.items():
    if asins:
        log_removal(asins, f'incomplete time series — NaN in {col} after ffill')

before_asins = df['ASIN'].nunique()
before_rows  = len(df)
df = df[~df['ASIN'].isin(asins_to_drop)].copy()

print(f'ASINs : {before_asins:,} → {df["ASIN"].nunique():,}  (dropped {before_asins - df["ASIN"].nunique():,})')
print(f'Rows  : {before_rows:,} → {len(df):,}  (dropped {before_rows - len(df):,})')
print()
print('NaN check after drop:')
for col in REQUIRED_COMPLETE:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  {"✅" if n==0 else "⚠️ "}')

ASINs : 79 → 65  (dropped 14)
Rows  : 9,006 → 7,410  (dropped 1,596)

NaN check after drop:
  SALES_RANK                          : 0  ✅
  PRICE                               : 0  ✅
  BUYBOX_PRICE                        : 0  ✅
  RATING                              : 0  ✅
  REVIEW_COUNT                        : 0  ✅


## ⑬ Reorder Columns — Paper Schema

In [52]:
CORE_COLS = [
    'ASIN', 'window', 'date',
    'SALES_RANK', 'is_SALES_RANK_filled',
    'PRICE', 'is_PRICE_filled',
    'BUYBOX_PRICE', 'is_BUYBOX_PRICE_filled',
    'text',
    'RATING', 'is_RATING_filled',
    'REVIEW_COUNT', 'is_REVIEW_COUNT_filled',
    'subcat', 'subcat_aggregated',
    'New Offer Count: Current',
    'Count of retrieved live offers: New, FBA',
    'Count of retrieved live offers: New, FBM',
    'Lightning Deals: Upcoming Deal',
    'Buy Box: Is FBA',
    'image',
]

missing = [c for c in CORE_COLS if c not in df.columns]
if missing:
    print(f'⚠️  Missing: {missing}')
else:
    df = df[CORE_COLS]
    print(f'✅ {len(df.columns)} columns in correct order')
    for i, col in enumerate(df.columns):
        print(f'  [{i:02d}] {col}')

✅ 22 columns in correct order
  [00] ASIN
  [01] window
  [02] date
  [03] SALES_RANK
  [04] is_SALES_RANK_filled
  [05] PRICE
  [06] is_PRICE_filled
  [07] BUYBOX_PRICE
  [08] is_BUYBOX_PRICE_filled
  [09] text
  [10] RATING
  [11] is_RATING_filled
  [12] REVIEW_COUNT
  [13] is_REVIEW_COUNT_filled
  [14] subcat
  [15] subcat_aggregated
  [16] New Offer Count: Current
  [17] Count of retrieved live offers: New, FBA
  [18] Count of retrieved live offers: New, FBM
  [19] Lightning Deals: Upcoming Deal
  [20] Buy Box: Is FBA
  [21] image


## ⑭ Schema Verification

In [53]:
print(f'{"#":<4} {"Column":<50} {"Expected":>16}  {"Actual":>16}')
print('-' * 94)
all_ok = True
for i, col in enumerate(df.columns):
    expected = PAPER_SCHEMA.get(col, 'object')
    actual   = str(df[col].dtype)
    # flag columns are int64 — expected is not in PAPER_SCHEMA, treat as OK
    ok = expected in actual or actual in expected or col.startswith('is_')
    mark = '✅' if ok else '❌'
    if not ok: all_ok = False
    print(f'{i:<4} {col:<50} {expected:>16}  {actual:>16}  {mark}')
print()
print('✅ All dtypes match!' if all_ok else '❌ Some dtypes do not match.')

#    Column                                                     Expected            Actual
----------------------------------------------------------------------------------------------
0    ASIN                                                         object            object  ✅
1    window                                                      float64           float64  ✅
2    date                                                 datetime64[ns]    datetime64[ns]  ✅
3    SALES_RANK                                                  float64           float64  ✅
4    is_SALES_RANK_filled                                         object             int64  ✅
5    PRICE                                                       float64           float64  ✅
6    is_PRICE_filled                                              object             int64  ✅
7    BUYBOX_PRICE                                                float64           float64  ✅
8    is_BUYBOX_PRICE_filled                                   

## ⑮ NaN Final Check

In [54]:
print(f'Shape : {df.shape}')
print(f'ASINs : {df["ASIN"].nunique():,}')
print()
nan_s = df.isna().sum().reset_index()
nan_s.columns = ['column', 'nan_count']
nan_s['nan_pct'] = (nan_s['nan_count'] / len(df) * 100).round(2)
nan_s['status']  = nan_s['nan_pct'].apply(
    lambda x: '🔴 HIGH' if x > 50 else ('🟡 MED' if x > 15 else '✅ OK')
)
print(nan_s.to_string(index=False))
remaining = nan_s[nan_s['nan_count'] > 0]
if len(remaining) == 0:
    print('\n✅ Zero NaN — all columns complete.')
else:
    print(f'\n⚠️  Remaining NaN in {len(remaining)} column(s) — review above.')

Shape : (7410, 22)
ASINs : 65

                                  column  nan_count  nan_pct status
                                    ASIN          0      0.0   ✅ OK
                                  window          0      0.0   ✅ OK
                                    date          0      0.0   ✅ OK
                              SALES_RANK          0      0.0   ✅ OK
                    is_SALES_RANK_filled          0      0.0   ✅ OK
                                   PRICE          0      0.0   ✅ OK
                         is_PRICE_filled          0      0.0   ✅ OK
                            BUYBOX_PRICE          0      0.0   ✅ OK
                  is_BUYBOX_PRICE_filled          0      0.0   ✅ OK
                                    text          0      0.0   ✅ OK
                                  RATING          0      0.0   ✅ OK
                        is_RATING_filled          0      0.0   ✅ OK
                            REVIEW_COUNT          0      0.0   ✅ OK
                 

## ⑯ Drop NaN Flag Columns and Save Full Cleaned Dataset

All cleaning is complete. Save all clean ASINs in this bucket as a single parquet.
Train/val split is applied in the next section using the pre-computed `asin_split.csv`.

In [55]:
# Drop NaN flag columns before saving
flag_cols = [f'is_{c}_filled' for c in FFILL_COLS if f'is_{c}_filled' in df.columns]
df = df.drop(columns=flag_cols).copy()
print(f'Dropped {len(flag_cols)} NaN-flag columns')
print(f'Final columns ({len(df.columns)}):')
for i, col in enumerate(df.columns):
    print(f'  [{i:02d}] {col:<50} {str(df[col].dtype)}')

Dropped 5 NaN-flag columns
Final columns (17):
  [00] ASIN                                               object
  [01] window                                             float64
  [02] date                                               datetime64[ns]
  [03] SALES_RANK                                         float64
  [04] PRICE                                              float64
  [05] BUYBOX_PRICE                                       float64
  [06] text                                               object
  [07] RATING                                             float64
  [08] REVIEW_COUNT                                       float64
  [09] subcat                                             object
  [10] subcat_aggregated                                  object
  [11] New Offer Count: Current                           int64
  [12] Count of retrieved live offers: New, FBA           int64
  [13] Count of retrieved live offers: New, FBM           int64
  [14] Lightning Deals: Upcoming

## ⑰ Save Full Cleaned Parquet

In [56]:
os.makedirs(DATA_DIR, exist_ok=True)
CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEAN_FILE, index=False)

sz = os.path.getsize(CLEAN_FILE) / 1e6
print(f'✅ Saved: {CLEAN_FILE}')
print(f'   Rows  : {len(df):,}')
print(f'   ASINs : {df["ASIN"].nunique():,}')
print(f'   Size  : {sz:.1f} MB')
print()
print('subcat_aggregated distribution:')
sc = df.drop_duplicates('ASIN')['subcat_aggregated'].value_counts()
for name, cnt in sc.items():
    print(f'  {name:<35} {cnt:>5}  ({cnt/sc.sum()*100:.1f}%)')

✅ Saved: /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits/all_products_clean.parquet
   Rows  : 7,410
   ASINs : 65
   Size  : 0.2 MB

subcat_aggregated distribution:
  Golf                                   46  (70.8%)
  Road Running                            7  (10.8%)
  Wrestling                               6  (9.2%)
  Fashion Sneakers                        3  (4.6%)
  Skateboarding                           1  (1.5%)
  Fitness & Cross-Training                1  (1.5%)
  Tennis & Racquet Sports                 1  (1.5%)


## ⑱ Apply Pre-Computed Stratified Split

Load `code/{BRAND_BUCKET}/asin_split.csv` (produced by `brand_split/01_stratified_split.ipynb`)
and materialize train/val parquets + key CSVs.

ASINs dropped in part1's cleaning are simply absent from the materialized splits, so
totals may be slightly less than the 50/50 in `asin_split.csv`. The split assignment
for surviving ASINs is unchanged.

In [57]:
SPLIT_FILE = KEY_DIR / 'asin_split.csv'
assert SPLIT_FILE.exists(), (
    f'Missing {SPLIT_FILE}\n'
    f'Did you run brand_split/01_stratified_split.ipynb first?'
)

split_df = pd.read_csv(SPLIT_FILE)
train_asins = set(split_df.loc[split_df['split'] == 'train', 'ASIN'])
val_asins   = set(split_df.loc[split_df['split'] == 'val',   'ASIN'])

df_train = df[df['ASIN'].isin(train_asins)].copy()
df_val   = df[df['ASIN'].isin(val_asins  )].copy()

dropped_train = train_asins - set(df_train['ASIN'].unique())
dropped_val   = val_asins   - set(df_val['ASIN'].unique())

TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VAL_FILE   = DATA_DIR / 'validation-00000-of-00001.parquet'
df_train.to_parquet(TRAIN_FILE, index=False)
df_val.to_parquet(VAL_FILE,   index=False)

KEY_DIR.mkdir(parents=True, exist_ok=True)
for name, dfx in [('train', df_train), ('val', df_val)]:
    keys = dfx[['ASIN', 'date']].copy()
    keys['date'] = pd.to_datetime(keys['date']).dt.strftime('%Y-%m-%d')
    keys.to_csv(KEY_DIR / f'main_{name}_keys.csv', index=False)

print('=' * 60)
print('TRAIN / VAL SPLIT APPLIED')
print('=' * 60)
print(f'Split source : {SPLIT_FILE.relative_to(PROJECT_ROOT)}')
print(f'Train  : {len(df_train):,} rows | {df_train["ASIN"].nunique():,} ASINs  (cleaning dropped {len(dropped_train)} of split-assigned ASINs)')
print(f'Val    : {len(df_val):,} rows | {df_val["ASIN"].nunique():,} ASINs  (cleaning dropped {len(dropped_val)} of split-assigned ASINs)')
print()
print(f'✅ {TRAIN_FILE.relative_to(PROJECT_ROOT)}')
print(f'✅ {VAL_FILE.relative_to(PROJECT_ROOT)}')
print(f'✅ {(KEY_DIR / "main_train_keys.csv").relative_to(PROJECT_ROOT)}')
print(f'✅ {(KEY_DIR / "main_val_keys.csv").relative_to(PROJECT_ROOT)}')

TRAIN / VAL SPLIT APPLIED
Split source : data/men_adidas/keys/asin_split.csv
Train  : 5,928 rows | 52 ASINs  (cleaning dropped 12 of split-assigned ASINs)
Val    : 1,482 rows | 13 ASINs  (cleaning dropped 2 of split-assigned ASINs)

✅ data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits/train-00000-of-00001.parquet
✅ data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits/validation-00000-of-00001.parquet
✅ data/men_adidas/keys/main_train_keys.csv
✅ data/men_adidas/keys/main_val_keys.csv


## ⑲ Next Step

In [58]:
print('=' * 60)
print(f'PART 1 COMPLETE — {BRAND_BUCKET}')
print('=' * 60)
print(f'Cleaned     : {CLEAN_FILE.relative_to(PROJECT_ROOT)}')
print(f'              {df["ASIN"].nunique():,} clean ASINs × {df["date"].nunique()} dates × 2 windows')
print(f'Train       : {df_train["ASIN"].nunique():,} ASINs')
print(f'Val         : {df_val["ASIN"].nunique():,} ASINs')
print()
print('Next:')
print(f'  00_part2_download_images.ipynb  (~{df["ASIN"].nunique():,} images for this bucket)')
print('  00_part5_create_image_parquet.ipynb')
print('  00_part3a/b/c/d_train_*.ipynb   (training on train/val splits)')

PART 1 COMPLETE — men_adidas
Cleaned     : data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits/all_products_clean.parquet
              65 clean ASINs × 57 dates × 2 windows
Train       : 52 ASINs
Val         : 13 ASINs

Next:
  00_part2_download_images.ipynb  (~65 images for this bucket)
  00_part5_create_image_parquet.ipynb
  00_part3a/b/c/d_train_*.ipynb   (training on train/val splits)


## ⑳ Verify Saved File

In [59]:
df_chk = pd.read_parquet(CLEAN_FILE)
print('=' * 60)
print('FINAL VERIFICATION')
print('=' * 60)
print(f'Rows  : {len(df_chk):,}')
print(f'ASINs : {df_chk["ASIN"].nunique():,}')
print(f'Cols  : {len(df_chk.columns)}')
print(f'Date  : {df_chk["date"].min().date()} → {df_chk["date"].max().date()}')
print()
print('NaN check:')
all_clean = True
for col in ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']:
    n = df_chk[col].isna().sum()
    flag = '✅' if n == 0 else '⚠️ '
    if n > 0:
        all_clean = False
    print(f'  {col:<35} : {n:,}  {flag}')
print()
if all_clean:
    print('✅ all_products_clean.parquet verified.')
else:
    print('⚠️  Some NaN remain — review removal log.')

FINAL VERIFICATION
Rows  : 7,410
ASINs : 65
Cols  : 17
Date  : 2025-03-03 → 2026-03-30

NaN check:
  SALES_RANK                          : 0  ✅
  PRICE                               : 0  ✅
  BUYBOX_PRICE                        : 0  ✅
  RATING                              : 0  ✅
  REVIEW_COUNT                        : 0  ✅

✅ all_products_clean.parquet verified.


## ㉑ Complete ASIN Removal Log

In [60]:
log_df = pd.DataFrame(removal_log)

print('=' * 65)
print('COMPLETE ASIN REMOVAL LOG')
print('=' * 65)

if len(log_df) == 0:
    print(f'Total ASINs removed : 0  (clean bucket — nothing dropped)')
    print(f'Total ASINs kept    : {df["ASIN"].nunique():,}')
    print(f'Started with        : {df["ASIN"].nunique():,} ASINs')
else:
    print(f'Total ASINs removed : {log_df["ASIN"].nunique():,}')
    print(f'Total ASINs kept    : {df["ASIN"].nunique():,}')
    print(f'Started with        : {log_df["ASIN"].nunique() + df["ASIN"].nunique():,} ASINs')
    print()

    # Summary by reason
    summary = log_df.groupby('reason')['ASIN'].nunique().reset_index()
    summary.columns = ['reason', 'count']
    print('By reason:')
    print(summary.to_string(index=False))
    print()

    # Detailed list by reason
    for reason, group in log_df.groupby('reason'):
        asins = sorted(group['ASIN'].unique())
        print(f'── {reason}  ({len(asins)} ASINs) ──')
        for asin in asins:
            subcat = asin_subcat_map.get(asin, 'n/a')
            print(f'  {asin}  {subcat}')
        print()

COMPLETE ASIN REMOVAL LOG
Total ASINs removed : 14
Total ASINs kept    : 65
Started with        : 79 ASINs

By reason:
                                                  reason  count
incomplete time series — NaN in BUYBOX_PRICE after ffill      1
       incomplete time series — NaN in PRICE after ffill      8
      incomplete time series — NaN in RATING after ffill     11
incomplete time series — NaN in REVIEW_COUNT after ffill     11
  incomplete time series — NaN in SALES_RANK after ffill      5

── incomplete time series — NaN in BUYBOX_PRICE after ffill  (1 ASINs) ──
  B0CYR2Z2P1  Fashion Sneakers

── incomplete time series — NaN in PRICE after ffill  (8 ASINs) ──
  B07T391K1H  Football
  B08CZ3Z52Q  Road Running
  B0CYR2Z2P1  Fashion Sneakers
  B0CZ74HSD4  Fashion Sneakers
  B0D17WQJG2  Fashion Sneakers
  B0D2S77TY9  Basketball
  B0DJTSDSBV  Fitness & Cross-Training
  B0DJTYB32Z  Road Running

── incomplete time series — NaN in RATING after ffill  (11 ASINs) ──
  B0C62CZ47J  Golf
